# Gold · qualidade_dados

**Origem:** bronze, silver e `gold.chamados_nota` → **Destino:** `vcredit_gold.qualidade_tabelas` e `vcredit_gold.qualidade_indicadores`
**Grão:** uma linha por tabela / uma linha por inconsistência · **Depende de:** todas as silvers e `gold_chamados_nota`

Métricas do resultado dos tratamentos, para o painel: linhas por camada e contagem de cada inconsistência encontrada.

**Decisões:** as métricas são recalculadas a cada execução, para o painel refletir o estado atual do pipeline. Pipeline idempotente (sobrescreve as tabelas).

In [0]:
from pyspark.sql import functions as F

cat = "rocket_lab"
b, s, g = f"{cat}.vcredit_bronze", f"{cat}.vcredit_silver", f"{cat}.vcredit_gold"

# 1) linhas por camada
tabelas = ["canais", "motivos", "atendentes", "clientes", "custos", "pesquisa", "chamados", "chamados_hora"]
linhas = []
for t in tabelas:
    nb = spark.table(f"{b}.{t}").count()
    ns = spark.table(f"{s}.{t}").count()
    linhas.append((t, nb, ns, nb - ns, round((nb - ns) / nb * 100, 2) if nb else 0.0))

qt = spark.createDataFrame(linhas, "tabela string, linhas_bronze long, linhas_silver long, removidas long, pct_removidas double")
(qt.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable(f"{g}.qualidade_tabelas"))

# 2) inconsistências tratadas
ch = spark.table(f"{s}.chamados")
hr = spark.table(f"{s}.chamados_hora")
cn = spark.table(f"{g}.chamados_nota")
total = ch.count()

def ind(nome, tratamento, valor):
    return (nome, tratamento, int(valor), round(int(valor) / total * 100, 2))

ch_bronze = spark.table(f"{b}.chamados")
cu = spark.table(f"{s}.custos").filter("custo is not null").select("id_chamado").distinct()

indicadores = [
    ind("Chamados duplicados na origem", "Removidos (cópias idênticas)",
        ch_bronze.count() - total),
    ind("Chamados com horário impossível", "Removidos da chamados_hora",
        spark.table(f"{b}.chamados_hora").count() - hr.count()),
    ind("Abandonos (sem hora de início)", "Mantidos e marcados",
        hr.filter("sem_inicio_atendimento").count()),
    ind("Chamados sem custo", "Mantidos; ficam fora das contas de custo",
        ch.join(cu, "id_chamado", "left_anti").count()),
    ind("Cliente fora da base limpa de clientes", "Mantidos e marcados",
        hr.filter("not cliente_valido").count()),
    ind("Cliente diferente entre chamados e chamados_hora", "Mantidos",
        ch.select("id_chamado", F.col("id_cliente").alias("a"))
          .join(hr.select("id_chamado", F.col("id_cliente").alias("b")), "id_chamado")
          .filter("a != b").count()),
    ind("Atendente placeholder (0, 99, 999)", "Mantidos e marcados",
        ch.filter("flag_atendente_placeholder").count()),
    ind("Tempo de atendimento corrigido pelos horários", "Recalculado na gold",
        cn.filter("atend_corrigido").count()),
    ind("Tempo de espera corrigido pelos horários", "Recalculado na gold",
        cn.filter("espera_corrigida").count()),
    ind("Chamados com nota inválida na pesquisa (vazia, 0 ou > 5)", "Nota anulada",
        spark.table(f"{s}.pesquisa").filter("nota_atendimento is null")
             .select("id_chamado").distinct().count()),
]

qi = spark.createDataFrame(indicadores, "inconsistencia string, tratamento string, chamados long, pct_chamados double")
(qi.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable(f"{g}.qualidade_indicadores"))

display(spark.table(f"{g}.qualidade_tabelas"))
display(spark.table(f"{g}.qualidade_indicadores"))